# Clase 148 — LLMs aplicados: fine-tuning y prompting

Uso real de **Large Language Models** (Llama 3, Mistral, Qwen): prompting técnico
(zero-shot, few-shot, chain-of-thought), generación con `generate(...)`
(temperature / top_p), y **fine-tuning eficiente** con **LoRA** (`peft`).

**Requiere:** `transformers`, `torch`, `peft` (y descarga de un modelo causal).
Las partes de prompting y el muestreo (temperature / top_p / LoRA como
`ΔW = B·A`) se ilustran en **numpy ejecutable**; la carga del LLM va guardada.

## 1. Entorno

In [ ]:
try:
    import transformers, torch
    HAS_LLM = True
    print('transformers:', transformers.__version__)
except Exception as e:
    HAS_LLM = False
    print('LLM stack no instalado. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)

## 2. Prompting: zero-shot, few-shot y chain-of-thought

El prompt es puro texto estructurado. Los modelos *chat* usan una lista de
mensajes con roles (`system`, `user`, `assistant`) que el tokenizer convierte con
`apply_chat_template`.

In [ ]:
task = 'Clasificá la queja como: facturación, envío o producto.'

zero_shot = f"{task}\nQueja: 'Me cobraron dos veces este mes.'\nCategoría:"

few_shot = f"""{task}
Queja: 'El paquete llegó roto.' -> producto
Queja: 'Nunca me llegó el pedido.' -> envío
Queja: 'Me cobraron de más.' -> facturación
Queja: 'Me cobraron dos veces este mes.' ->"""

cot = few_shot + "\nPensá paso a paso antes de responder."

for name, p in [('zero-shot', zero_shot), ('few-shot', few_shot), ('CoT', cot)]:
    print(f'[{name}] {len(p)} chars')

messages = [
    {'role': 'system', 'content': 'Sos un clasificador de quejas conciso.'},
    {'role': 'user', 'content': zero_shot},
]
print('roles:', [m['role'] for m in messages])

## 3. Generación con `AutoModelForCausalLM.generate(...)`

`generate` acepta `temperature` (aleatoriedad), `top_p` (nucleus sampling) y
`max_new_tokens`. `do_sample=True` activa el muestreo (si no, greedy).

In [ ]:
if HAS_LLM:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    name = 'mistralai/Mistral-7B-Instruct-v0.2'
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16,
                                                 device_map='auto')
    prompt = tok.apply_chat_template(messages, tokenize=False,
                                     add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=64, do_sample=True,
                         temperature=0.7, top_p=0.9)
    print(tok.decode(out[0], skip_special_tokens=True))
else:
    print('generate(max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)')

## 4. Cómo actúan `temperature` y `top_p` (nucleus) sobre los logits

Ejecutable en numpy: `temperature` divide los logits antes del softmax; `top_p`
se queda con los tokens cuya masa acumulada llega a `p` y renormaliza.

In [ ]:
def softmax(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

logits = np.array([2.0, 1.0, 0.5, 0.1, -1.0, -2.0])

def sample_top_p(logits, temperature=1.0, top_p=0.9, rng=np.random.default_rng(0)):
    probs = softmax(logits / temperature)
    order = np.argsort(probs)[::-1]
    cum = np.cumsum(probs[order])
    keep = order[:np.searchsorted(cum, top_p) + 1]   # nucleo minimo
    p = probs[keep] / probs[keep].sum()
    return int(rng.choice(keep, p=p)), keep

for t in (0.3, 1.0):
    idx, keep = sample_top_p(logits, temperature=t, top_p=0.9)
    print(f'T={t}: token={idx} | nucleo(top_p=0.9)={keep.tolist()}')

## 5. LoRA: fine-tuning eficiente con `peft`

LoRA congela el modelo base y aprende un delta de bajo rango
`ΔW = B · A` con `A ∈ R^{r×d}`, `B ∈ R^{d×r}`. Con `r=16` se entrena ~0.5 % de
los parámetros. QLoRA = LoRA sobre un base cuantizado en 4-bit.

In [ ]:
# Ilustracion del ahorro de parametros (numpy, ejecutable)
d, r = 4096, 16
full = d * d                      # W completa
lora = 2 * d * r                  # A y B
print(f'W full: {full:,} params | LoRA(r={r}): {lora:,} | ratio={lora/full:.4%}')

if HAS_LLM:
    from peft import LoraConfig, get_peft_model
    cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                     task_type='CAUSAL_LM')
    peft_model = get_peft_model(model, cfg)
    peft_model.print_trainable_parameters()  # ~0.5 % entrenable
else:
    print('LoraConfig(r=16, target_modules=[q,k,v,o_proj]) -> get_peft_model(base, cfg)')

## 6. SFT con `Trainer` (o `trl.SFTTrainer`)

Sobre el modelo con adapters LoRA se entrena igual que un modelo normal: solo se
actualizan los pesos LoRA. Con `trl` el `SFTTrainer` simplifica el formateo de
instrucciones.

In [ ]:
if HAS_LLM:
    from transformers import TrainingArguments, Trainer
    args = TrainingArguments(output_dir='lora-out', num_train_epochs=3,
                             per_device_train_batch_size=4,
                             gradient_accumulation_steps=4,
                             learning_rate=2e-4, fp16=True, logging_steps=10)
    # trainer = Trainer(model=peft_model, args=args, train_dataset=train_ds)
    # trainer.train()   # solo se actualizan los adapters LoRA
    print('SFT LoRA configurado: lr=2e-4, 3 epocas, grad_accum=4.')
else:
    print('trl.SFTTrainer(model=peft_model, args=..., train_dataset=...).train()')

## Ejercicios

1. **Prompt engineering**: escribí 5 variantes (zero-shot, few-shot, CoT...) para
   una tarea de clasificación y compará su comportamiento en un set chico.
2. **Muestreo**: variá `temperature ∈ {0.2, 0.7, 1.2}` y `top_p ∈ {0.8, 0.95}` en
   la celda 4 y observá cómo cambia el núcleo y la aleatoriedad.
3. **LoRA**: cambiá `r ∈ {8, 16, 64}` en la celda 5 y calculá el ratio de
   parámetros entrenables.
4. **SFT**: adaptá la celda 6 para fine-tunear Mistral 7B Instruct con QLoRA
   (4-bit) sobre 500-2000 pares (instrucción, respuesta).

## Conclusiones

- El prompting (few-shot, chain-of-thought, system prompt) cambia mucho la calidad
  sin tocar los pesos.
- `generate(...)` con `temperature`/`top_p` controla el trade-off
  determinismo ↔ diversidad; `top_p` es nucleus sampling.
- LoRA aprende `ΔW = B·A` de bajo rango: se entrena <1 % de los parámetros.
- QLoRA (4-bit) permite fine-tunear un 7B en una GPU de 24 GB.
- El stack `transformers` + `peft` + `trl` cubre prompting, SFT y alineamiento.